# Week 3 — Coconut-curriculum GPT-2 training — latent-agent-cost

Trained comparison for *Tokens Are Not Dollars*: the Week-2 stage-1 pilot showed training-free latent feedback collapses on ProsQA (latent 0.014 vs explicit 0.202), so this run trains a small GPT-2 (124M) through Coconut's curriculum on the ProsQA train split's gold reasoning chains, then the trained checkpoint is evaluated separately through the harness's existing `gpt2_cot` / `gpt2_latent` pair (see `RUNBOOK.md`, Week 3).

This notebook does training only:
- Data prep: `scripts/prepare_prosqa_train.py` (canonical ProsQA train split, 17,886 validated gold chains)
- Training: `scripts/train_coconut_gpt2.py` — stages 0,1,2,3 x 1 epoch, lr 1e-4, grad-accum 8, `--save-each-stage`

**Resumability:** `--save-each-stage` writes `checkpoints/coconut-gpt2-prosqa-stage<k>` after each stage, so a session killed mid-curriculum resumes from the last stage checkpoint (`--base-model` accepts the local path) instead of restarting. Estimated 5-8 GPU-hours total — one Kaggle session, inside the ~12 h cap.

**Before running:** notebook Settings -> **Accelerator = GPU** (T4 x2 or P100) and **Internet = ON** (needed for the git clone, the HF dataset downloads, and the model download). Use **Save Version -> Save & Run All (Commit)** so the checkpoints persist in the Output.

**Evaluation note (do NOT change):** when the trained checkpoint is evaluated, `--latent-steps` MUST be 3 — the final trained stage — per `RUNBOOK.md`.

In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU visible. Set Accelerator = GPU in the notebook settings and re-run.")
print("device:", torch.cuda.get_device_name(0))

Clone the project (fresh every run) and install its dependencies. `requirements.txt` already includes `tokenecon==0.2.0`; the one local pin filtered out is `torch` — Kaggle's preinstalled CUDA build must be kept, never replaced by the local CPU pin.

In [ ]:
!rm -rf latent-agent-cost
!git clone https://github.com/karmendra8386/latent-agent-cost
%cd latent-agent-cost
!grep -v '^torch' requirements.txt > /tmp/req-kaggle.txt
!pip install -q -r /tmp/req-kaggle.txt
!python -c "import torch; print('after install - torch:', torch.__version__, '| cuda:', torch.cuda.is_available())"

**Data prep** — build the ProsQA training file from the canonical train split. The prep script validates each gold `steps` chain mechanically (0 dropped in the local validation run).

In [ ]:
!python scripts/prepare_prosqa_train.py --split train --out data/prosqa_train.jsonl
!wc -l data/prosqa_train.jsonl

**Training** — Coconut curriculum, 4 stages x 1 epoch, per-stage checkpoints saved for resumability (exact RUNBOOK Week-3 command).

In [ ]:
!python scripts/train_coconut_gpt2.py --data data/prosqa_train.jsonl --base-model openai-community/gpt2 --out checkpoints/coconut-gpt2-prosqa --stages 0,1,2,3 --epochs-per-stage 1 --lr 1e-4 --grad-accum 8 --device cuda --save-each-stage

List the checkpoints produced — the final `coconut-gpt2-prosqa` directory plus one `coconut-gpt2-prosqa-stage<k>` per completed stage. These persist in the run's Output at session end; if the session was killed, resume training from the highest stage checkpoint.

In [ ]:
!ls -la checkpoints/
!du -sh checkpoints/*